# N166 · 屏障、生产消费与有界队列

**目标：** 维护容量约束和批次组成而不跨批串线。

**先修：** N165, N053。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Barrier；生产者/消费者；not_empty/not_full；容量不变量；批次同步。

## 从问题到算法

H2O 不要求三个原子的内部次序为 HHO，但每个连续三原子批次必须恰好两个 H、一个 O。信号量限制成分，屏障等待本批三次回调全部完成后才释放下一批许可证。

有界队列的关键是两个条件：满时生产者等待，空时消费者等待。检查条件、修改队列和通知都在同一互斥区；wait 会释放锁，使能够改变条件的对方有机会前进。

队列可选记录真实操作轨迹，用于显示占用与检查线性化次序；默认不记录，避免教学监测让正常队列空间随操作次数增长。

## 最小实现

**本章接口：** `H2O`、`BoundedBlockingQueue.enqueue/dequeue/size`

In [1]:
THREAD_SOURCE = r'''import threading
from collections import deque

def run_threads(functions):
    """Caller must run potentially blocking cases inside the isolated process."""
    errors = []
    error_lock = threading.Lock()
    def guarded(fn):
        try:
            fn()
        except BaseException as error:
            with error_lock:
                errors.append(error)
    threads = [threading.Thread(target=guarded, args=(fn,)) for fn in functions]
    for thread in threads:
        thread.start()
    for thread in threads:
        thread.join()
    if errors:
        raise errors[0]
    assert not any(thread.is_alive() for thread in threads)

class H2O:
    def __init__(self):
        self.hydrogen_slots = threading.Semaphore(2)
        self.oxygen_slots = threading.Semaphore(1)
        self.molecule_done = threading.Barrier(3)
    def hydrogen(self, releaseHydrogen):
        self.hydrogen_slots.acquire()
        releaseHydrogen()
        self.molecule_done.wait()
        self.hydrogen_slots.release()
    def oxygen(self, releaseOxygen):
        self.oxygen_slots.acquire()
        releaseOxygen()
        self.molecule_done.wait()
        self.oxygen_slots.release()

class BoundedBlockingQueue:
    def __init__(self,capacity,record_trace=False):
        if capacity <= 0:
            raise ValueError('Capacity must be positive')
        self.capacity = capacity
        self.items = deque()
        self.condition = threading.Condition()
        self.history = [] if record_trace else None
    def _record(self,action,value):
        if self.history is not None:
            self.history.append((action,value,len(self.items)))
    def enqueue(self,element):
        with self.condition:
            while len(self.items) == self.capacity:
                self.condition.wait()
            self.items.append(element)
            self._record('enqueue',element)
            self.condition.notify_all()
    def dequeue(self):
        with self.condition:
            while not self.items:
                self.condition.wait()
            element = self.items.popleft()
            self._record('dequeue',element)
            self.condition.notify_all()
            return element
    def size(self):
        with self.condition:
            return len(self.items)
'''

import json
import subprocess
import sys

# This defines the visible APIs only; it does not launch a thread.
exec(THREAD_SOURCE, globals())

def run_isolated(extra):
    """A timeout is a failure; subprocess.run kills and reaps the child."""
    result = subprocess.run([sys.executable, '-c', THREAD_SOURCE + '\n' + extra],
        text=True, capture_output=True, check=True, timeout=20)
    return result.stdout

## 正确性、前提与复杂度

屏障释放前已有本批全部三个回调，因此下一批不会插入本批输出中。队列在锁内 append/popleft，分别是入队与出队的线性化点；长度不变量由满/空 while 检查保持。队列完成需要生产消费数量匹配；水分子需要原子计量匹配，这些不是超时后偷偷忽略的输入。

**代价：** 队列单次结构操作 O(1)，默认存储 O(capacity)，诊断轨迹另占 O(操作次数)。同步等待时长依赖负载和调度；屏障/许可证状态 O(1)。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

rows = json.loads(run_isolated("""import json
q=BoundedBlockingQueue(2,record_trace=True)
def produce():
    for value in range(6):q.enqueue(value)
def consume():
    for _ in range(6):q.dequeue()
run_threads([consume,produce])
print(json.dumps(q.history))
"""))
show_table(['operation','value','queue size after operation'],rows)

operation,value,queue size after operation
enqueue,0,1
enqueue,1,2
dequeue,0,1
dequeue,1,0
enqueue,2,1
enqueue,3,2
dequeue,2,1
dequeue,3,0
enqueue,4,1
enqueue,5,2


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
print(run_isolated("""import itertools,random
from collections import Counter
for arrival in sorted(set(itertools.permutations('HHHHOO'))):
    water=H2O();out=[];lock=threading.Lock()
    def append(atom):
        with lock:out.append(atom)
    functions=[(lambda:water.hydrogen(lambda:append('H'))) if atom=='H'
               else (lambda:water.oxygen(lambda:append('O'))) for atom in arrival]
    run_threads(functions)
    assert len(out)==6
    assert all(Counter(out[i:i+3])==Counter('HHO') for i in range(0,6,3))
for capacity in [1,2,5]:
    q=BoundedBlockingQueue(capacity,record_trace=True)
    outputs=[];out_lock=threading.Lock()
    def producer(p):
        for i in range(12):q.enqueue((p,i))
    def consumer():
        for _ in range(12):
            value=q.dequeue()
            with out_lock:outputs.append(value)
    functions=[lambda p=p:producer(p) for p in range(3)]+[consumer for _ in range(3)]
    random.Random(capacity).shuffle(functions)
    run_threads(functions)
    assert Counter(outputs)==Counter((p,i) for p in range(3) for i in range(12))
    assert q.size()==0
    assert all(0<=size<=capacity for _,_,size in q.history)
    enqueued=[value for action,value,size in q.history if action=='enqueue']
    dequeued=[value for action,value,size in q.history if action=='dequeue']
    assert enqueued==dequeued
assert threading.active_count()==1
print('Molecule batches, FIFO linearization, capacity and exact delivery passed.')
"""))
print('N166: 所有本章断言通过')

Molecule batches, FIFO linearization, capacity and exact delivery passed.

N166: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 区分可选输出顺序和每批2H1O组成约束。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明多个条件共享同一锁。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1117. Building H2O（canonical）
- 1188. Design Bounded Blocking Queue（extension）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。